# LIBRAIRIES

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from affine import Affine
from scipy.ndimage import gaussian_filter
from rasterio.features import rasterize, geometry_mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as mpatches
import fiona
import rasterio
from rasterio.enums import MergeAlg
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point
import libpysal
import esda
import rasterstats
import warnings
warnings.filterwarnings("ignore")
import math
from statsmodels.nonparametric.smoothers_lowess import lowess
import matplotlib.cm as cm
%config InlineBackend.figure_format = 'png'

import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from utils.config import *
from utils.functions import *

# ── Force reload ────────────────────────────────────────
import importlib
import utils.config as cfg
import utils.functions as fn
importlib.reload(cfg)
importlib.reload(fn)

# PATH

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

# ─── Paths ──────────────────────────────────────────────────────────────────
input_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/INPUT/"
output_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/"
input_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/INPUT/"
output_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/OUTPUT/"
output_file_path_PL_RASTER = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/RASTER/"

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/GE/step-1/'
output_step2_path='../../Data/output/GE/step-2/'
output_step3_path='../../Data/output/GE/step-3/'

# IMPORTS

In [ ]:
canton_GE  = gpd.read_file(f'{input_file_path}/network_agreg/CANTON_GE/CANTON_POLYGON.shp')
canton_GE = canton_GE.to_crs(operation_crs)

legs_GE_walk_regular = gpd.read_parquet(f'{output_file_path_PL}legs_GE_walk_regular.parquet')
zones_girec = gpd.read_file(f'{output_step3_path}/step3_aggregated_index_girec.gpkg')
zones_girec = zones_girec.to_crs(operation_crs)
carreau_200 = gpd.read_file(f'{output_step3_path}/step3_aggregated_index_carreau200.gpkg')
carreau_200 = carreau_200.to_crs(operation_crs)
communes  = gpd.read_file(input_file_path_PL + "CAD_COMMUNE-SHP/CAD_COMMUNES_GE_fusionnee.shp").to_crs(epsg=2056)
lac_leman = gpd.read_file(input_file_path_PL + "LAC_LEMAN_WITHOUT_BRIDGE-SHP/LAC_LEMAN_WITHOUT_BRIDGE.shp").to_crs(epsg=2056)

In [ ]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")

In [ ]:
users_GE_walk_regular = pd.read_csv(f'{output_file_path_PL}users_GE_walk_regular.csv')

In [ ]:
zones_communes_GE_fusionnee = gpd.read_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee.parquet')
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.to_crs(operation_crs)

In [ ]:
# ─── Chargement des rasters exportés ─────────────────────────────────────────
raster_mean_clipped_v2 = load_raster(output_file_path_PL_RASTER, "density_all")

rasters_gender = {g: load_raster(output_file_path_PL_RASTER, f"density_{g}")
                  for g, _ in gender_filters}

rasters_age    = {a: load_raster(output_file_path_PL_RASTER, f"density_{a}")
                  for a, _ in age_filters}

rasters_income = {i: load_raster(output_file_path_PL_RASTER, f"density_{i}")
                  for i, _ in income_filters}

rasters_car    = {c: load_raster(output_file_path_PL_RASTER, f"density_{c}")
                  for c, _ in car_filters}

rasters_tp     = {t: load_raster(output_file_path_PL_RASTER, f"density_{t}")
                  for t, _ in tp_filters}

rasters_hourly = {s: load_raster(output_file_path_PL_RASTER, f"density_{s}")
                  for s, _, _, _, _ in time_slots}

# ─── Vérification ─────────────────────────────────────────────────────────────
print(f"✓ raster_mean_clipped_v2 : {raster_mean_clipped_v2.shape}")
print(f"✓ rasters_gender         : {list(rasters_gender.keys())}")
print(f"✓ rasters_age            : {list(rasters_age.keys())}")
print(f"✓ rasters_income         : {list(rasters_income.keys())}")
print(f"✓ rasters_car            : {list(rasters_car.keys())}")
print(f"✓ rasters_tp             : {list(rasters_tp.keys())}")
print(f"✓ rasters_hourly         : {list(rasters_hourly.keys())}")

In [ ]:
# ─── Chargement des rasters horaires ─────────────────────────────────────────
hourly_slots = [("all_day", 0, 0, 23, 59)] + [
    (f"{h:02d}h", h, 0, h+1, 0) for h in range(6, 22)
]
hourly_names = [s for s, *_ in hourly_slots if s != "all_day"]
hourly_labels = {f"{h:02d}h": f"{h:02d}h - {h+1:02d}h" for h in range(6, 22)}
hourly_labels["all_day"] = "All day"

rasters_hourly_h = {s: load_raster(output_file_path_PL_RASTER, f"density_hourly_{s}")
                    for s in hourly_names + ["all_day"]}

# ─── Normalisation commune ────────────────────────────────────────────────────
hourly_h_max = compute_scale(
    [rasters_hourly_h[s] for s in hourly_names if (rasters_hourly_h[s] > 0).any()],
    clip_percentile, mode=norm_mode
)

print(f"✓ {len(rasters_hourly_h)} rasters horaires chargés")
print(f"hourly_h_max : {hourly_h_max:.6f}")

In [ ]:
# ─── Reprojection en EPSG:2056 ────────────────────────────────────────────────
legs_GE_walk_regular = legs_GE_walk_regular.to_crs(epsg=2056)
canton_GE            = canton_GE.to_crs(epsg=2056)

print(f"CRS legs   : {legs_GE_walk_regular.crs}")
print(f"CRS canton : {canton_GE.crs}")

# ─── Paramètres raster ────────────────────────────────────────────────────────
pixel_size = 10

xmin, ymin, xmax, ymax = canton_GE.total_bounds

raster_width  = int((xmax - xmin) / pixel_size)
raster_height = int((ymax - ymin) / pixel_size)

transform = Affine(pixel_size, 0, xmin,
                   0, -pixel_size, ymax)

print(f"total_bounds  : {xmin:.0f}, {ymin:.0f}, {xmax:.0f}, {ymax:.0f}")
print(f"raster_width  : {raster_width}")
print(f"raster_height : {raster_height}")

# ─── Masque canton ────────────────────────────────────────────────────────────
canton_mask = geometry_mask(
    geometries=canton_GE.geometry,
    out_shape=(raster_height, raster_width),
    transform=transform,
    invert=True
)

lac_mask = geometry_mask(
    geometries=lac_leman.geometry,
    out_shape=(raster_height, raster_width),
    transform=transform,
    invert=True
)

canton_GE_mask = canton_mask & ~lac_mask

print(f"pixels canton  : {canton_GE_mask.sum():,}")

In [ ]:
# ─── Extent ───────────────────────────────────────────────────────────────────
extent = [xmin, xmax, ymin, ymax]  

# ─── Girec ────────────────────────────────────────────────────────────────────
girec = gpd.read_file(input_file_path_PL + "GEO_GIREC-SHP/GEO_GIREC.shp").to_crs(epsg=2056)

# ─── Focus commune ────────────────────────────────────────────────────────────
focus_commune_name = "Genève"
focus_commune      = communes[communes["COMMUNE"] == focus_commune_name]
xmin_z, ymin_z, xmax_z, ymax_z = focus_commune.total_bounds
margin = 200

# STAT ON DIFF RASTER

In [ ]:
# ─── Raster all users (référence commune) ────────────────────────────────────
rasters_gender_with_all = {"all": raster_mean_clipped_v2, **rasters_gender}
rasters_age_with_all    = {"all": raster_mean_clipped_v2, **rasters_age}
rasters_income_with_all = {"all": raster_mean_clipped_v2, **rasters_income}
rasters_car_with_all    = {"all": raster_mean_clipped_v2, **rasters_car}
rasters_tp_with_all     = {"all": raster_mean_clipped_v2, **rasters_tp}


gender_filters_with_all = [("all", None)] + [f for f in gender_filters if f[0] != "all"]
age_filters_with_all    = [("all", None)] + list(age_filters)
income_filters_with_all = [("all", None)] + list(income_filters)
car_filters_with_all    = [("all", None)] + list(car_filters)
tp_filters_with_all     = [("all", None)] + list(tp_filters)


# ─── Calcul des stats ─────────────────────────────────────────────────────────
df_stats_gender, _ = compute_group_stats(rasters_gender_with_all, gender_filters_with_all, canton_GE_mask)
df_stats_age,    _ = compute_group_stats(rasters_age_with_all,    age_filters_with_all,    canton_GE_mask)
df_stats_income, _ = compute_group_stats(rasters_income_with_all, income_filters_with_all, canton_GE_mask)
df_stats_car,    _ = compute_group_stats(rasters_car_with_all,    car_filters_with_all,    canton_GE_mask)
df_stats_tp,     _ = compute_group_stats(rasters_tp_with_all,     tp_filters_with_all,     canton_GE_mask)


# ─── Affichage ────────────────────────────────────────────────────────────────
for label, df in [
    ("Gender",       df_stats_gender),
    ("Age",          df_stats_age),
    ("Income",       df_stats_income),
    ("Car ownership",df_stats_car),
    ("PT subscription", df_stats_tp),

]:
    print(f"\n{'═'*60}")
    print(f"── Stats {label}")
    print(f"{'═'*60}")

    print("\n  ⚠ Active pixels only — DO NOT compare between groups")
    print(df[["mean_active", "median_active", "p90_active", "max_active",
              "std_active", "cv_active",
              "ratio_p90_median", "pct_freq_top10pct"]].round(6).to_string())

    print("\n  ✓ All canton pixels — comparable between groups")
    print(df[["mean_canton", "std_canton", "cv_canton",
              "pct_canton_couvert", "pct_zero_canton",
              "intensite_relative"]].round(6).to_string())

In [ ]:
plot_group_stats(df_stats_gender,  "Descriptive Statistics — Gender",         label_map=labels_all_groups, show_active_metrics=False)
plot_group_stats(df_stats_age,     "Descriptive Statistics — Age",             label_map=labels_all_groups, show_active_metrics=False)
plot_group_stats(df_stats_income,  "Descriptive Statistics — Income",          label_map=labels_all_groups, show_active_metrics=False)
plot_group_stats(df_stats_car,     "Descriptive Statistics — Car ownership",   label_map=labels_all_groups, show_active_metrics=False)
plot_group_stats(df_stats_tp,      "Descriptive Statistics — PT subscription", label_map=labels_all_groups, show_active_metrics=False)

# SPEARMAN CORRELATION

In [ ]:
# ─── Corrélation genre ────────────────────────────────────────────────────────
df_corr_gender, df_pval_gender = compute_spatial_correlation(
    rasters_gender_with_all, gender_filters_with_all, canton_GE_mask
)
plot_correlation_matrix(
    df_corr_gender, df_pval_gender,
    "Spearman Spatial Correlation — Gender",
    group_labels={"all": "All users", **gender_labels}
)
print_correlation_summary(
    df_corr_gender, df_pval_gender,
    "Spatial Correlation — Gender",
    group_labels={"all": "All users", **gender_labels}
)

# ─── Corrélation âge ──────────────────────────────────────────────────────────
df_corr_age, df_pval_age = compute_spatial_correlation(
    rasters_age_with_all, age_filters_with_all, canton_GE_mask
)
plot_correlation_matrix(
    df_corr_age, df_pval_age,
    "Spearman Spatial Correlation — Age",
    group_labels={"all": "All users", **age_labels}
)
print_correlation_summary(
    df_corr_age, df_pval_age,
    "Spatial Correlation — Age",
    group_labels={"all": "All users", **age_labels}
)

# ─── Corrélation revenu ───────────────────────────────────────────────────────
df_corr_income, df_pval_income = compute_spatial_correlation(
    rasters_income_with_all, income_filters_with_all, canton_GE_mask
)
plot_correlation_matrix(
    df_corr_income, df_pval_income,
    "Spearman Spatial Correlation — Income",
    group_labels={"all": "All users", **income_labels}  # ← income_labels au lieu de {i: i for ...}
)
print_correlation_summary(
    df_corr_income, df_pval_income,
    "Spatial Correlation — Income",
    group_labels={"all": "All users", **income_labels}
)

# ─── Corrélation voiture ──────────────────────────────────────────────────────
df_corr_car, df_pval_car = compute_spatial_correlation(
    rasters_car_with_all, car_filters_with_all, canton_GE_mask
)
plot_correlation_matrix(
    df_corr_car, df_pval_car,
    "Spearman Spatial Correlation — Car ownership",
    group_labels={"all": "All users", **car_labels}
)
print_correlation_summary(
    df_corr_car, df_pval_car,
    "Spatial Correlation — Car ownership",
    group_labels={"all": "All users", **car_labels}
)

# ─── Corrélation TP ──────────────────────────────────────────────────────
df_corr_tp, df_pval_tp = compute_spatial_correlation(
    rasters_tp_with_all, tp_filters_with_all, canton_GE_mask
)
plot_correlation_matrix(
    df_corr_tp, df_pval_tp,
    "Spearman Spatial Correlation — PT Subscription",
    group_labels={"all": "All users", **tp_labels}
)
print_correlation_summary(
    df_corr_tp, df_pval_tp,
    "Spatial Correlation — PT Subscription",
    group_labels={"all": "All users", **tp_labels}
)

In [ ]:
# ─── Grande matrice de corrélation — tous les groupes ────────────────────────
rasters_all_groups = {
    #"all"          : raster_mean_clipped_v2,
    # ─── Gender ───────────────────────────────────────────────────────────────
    "homme"        : rasters_gender["homme"],
    "femme"        : rasters_gender["femme"],
    # ─── Age ──────────────────────────────────────────────────────────────────
    "18-29"        : rasters_age["18-29"],
    "30-44"        : rasters_age["30-44"],
    "45-59"        : rasters_age["45-59"],
    "60+"          : rasters_age["60+"],
    # ─── Income ───────────────────────────────────────────────────────────────
    "tres_modeste" : rasters_income["tres_modeste"],
    "modeste"      : rasters_income["modeste"],
    #"median"       : rasters_income["median"],
    "aise"         : rasters_income["aise"],
    # ─── Car ownership ────────────────────────────────────────────────────────
    "no_car"       : rasters_car["no_car"],
    "has_car"      : rasters_car["has_car"],
    # ─── PT subscription ──────────────────────────────────────────────────────
    "no_tp"        : rasters_tp["no_tp"],
    "partial_tp"   : rasters_tp["partial_tp"],
    "full_tp"      : rasters_tp["full_tp"],
}

filters_all_groups = [(k, None) for k in rasters_all_groups.keys()]

# ─── Calcul ───────────────────────────────────────────────────────────────────
df_corr_full, df_pval_full = compute_spatial_correlation(
    rasters_dict  = rasters_all_groups,
    group_filters = filters_all_groups,
    canton_mask   = canton_GE_mask,
    method        = "spearman"
)

# ─── Plot ─────────────────────────────────────────────────────────────────────
plot_correlation_matrix(
    df_corr_full, df_pval_full,
    title        = "", #"Spearman Spatial Correlation — All groups",
    group_labels = labels_all_groups
)

print_correlation_summary(
    df_corr_full, df_pval_full,
    title        = "Spatial Correlation — All groups",
    group_labels = labels_all_groups
)

# AGGREGATE RASTER TO AREA - GIREC

In [ ]:
print("── Agrégation All users ──────────────────────────────")
aggregate_raster_to_area(zones_girec, {"all": raster_mean_clipped_v2}, [("all", None)], transform)

print("\n── Agrégation Gender ─────────────────────────────────")
aggregate_raster_to_area(zones_girec, rasters_gender, gender_filters, transform)

print("\n── Agrégation Age ────────────────────────────────────")
aggregate_raster_to_area(zones_girec, rasters_age, age_filters, transform)

print("\n── Agrégation Income ─────────────────────────────────")
aggregate_raster_to_area(zones_girec, rasters_income, income_filters, transform)

print("\n── Agrégation Car ownership ──────────────────────────")
aggregate_raster_to_area(zones_girec, rasters_car, car_filters, transform)

print("\n── Agrégation PT Subscription ────────────────────────")
aggregate_raster_to_area(zones_girec, rasters_tp, tp_filters, transform)

# ─── Vérification ─────────────────────────────────────────────────────────────
density_cols = [c for c in zones_girec.columns if c.startswith("density_")]
print(f"\n✓ Colonnes density ajoutées : {density_cols}")
print(f"✓ Zones GIREC total         : {len(zones_girec)}")
print(zones_girec[density_cols].describe().round(6))

### MORAN'S I + LISA

### LISA X WALK_INDEX

### DASHBOARD MORAN'S I | LISA | WALK_INDEX

#### NEW VERSION

In [ ]:
# ─── Vérification des variables socio-spatiales ───────────────────────────────
cols_check = ["freq_crim_norm", "desserte_score", "precarity_norm", "density_all", "walk_index"]

print(zones_girec[cols_check].describe().round(4))
print(f"\nNaN par colonne :")
print(zones_girec[cols_check].isna().sum())

# SPATIAL CORRELATION SPE AREA

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

# ─── Variables socio-spatiales ────────────────────────────────────────────────
socio_vars = {
    "freq_crim_norm" : "Crime frequency (mean)",
    "desserte_score" : "Transit accessibility score",
    "precarity_norm" : "Precarity index",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Pedestrian density vs socio-spatial variables\n Spearman correlation - All user — GIREC zones",
             fontsize=13, fontweight="bold")

for i, (col, label) in enumerate(socio_vars.items()):

    # ─── Données valides ──────────────────────────────────────────────────────
    df_plot = zones_girec[["density_all", col]].dropna()
    df_plot = df_plot[df_plot["density_all"] > 0]  # ← zones avec fréquentation

    x = df_plot[col].values
    y = df_plot["density_all"].values

    # ─── Corrélation de Spearman ──────────────────────────────────────────────
    from scipy.stats import spearmanr
    r, p = spearmanr(x, y)
    sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

    # ─── Scatter ──────────────────────────────────────────────────────────────────
    axes[i].scatter(x, y, alpha=0.4, s=20, color="#4C72B0", edgecolors="none")

    # ─── LOWESS sur log(y) ────────────────────────────────────────────────────────
    log_y        = np.log1p(y)
    lowess_vals  = lowess(log_y, x, frac=0.4)
    axes[i].plot(lowess_vals[:, 0], np.expm1(lowess_vals[:, 1]),
                color="red", linewidth=2, label="LOWESS")

    axes[i].set_yscale('log')
    #axes[i].set_ylabel("Mean daily pedestrian density\n(legs · day⁻¹ · user⁻¹) — log scale", fontsize=9)

    axes[i].set_xlabel(label, fontsize=10)
    axes[i].set_ylabel("Mean daily pedestrian density\n(legs · day⁻¹ · user⁻¹) - log scale", fontsize=9)
    axes[i].set_title(f"r={r:.3f} {sig} | n={len(df_plot)}", fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Groupes à comparer ───────────────────────────────────────────────────────
groups_to_plot = [
    #("all",          "All users",      "#333333", raster_mean_clipped_v2),
    ("homme",        "Men",            "#4C72B0", rasters_gender["homme"]),
    ("femme",        "Women",          "#DD8452", rasters_gender["femme"]),
    ("18-29",        "18-29",          "#55A868", rasters_age["18-29"]),
    ("60+",          "60+",            "#C44E52", rasters_age["60+"]),
    ("no_car",       "No car",         "#8172B2", rasters_car["no_car"]),
    ("has_car",      "Car owner",      "#937860", rasters_car["has_car"]),
    ("no_tp",        "No PT sub.",     "#DA8BC3", rasters_tp["no_tp"]),
    ("full_tp",      "Full PT sub.",   "#8C8C8C", rasters_tp["full_tp"]),
    ("tres_modeste", "Low income",     "#2196F3", rasters_income["tres_modeste"]),
    ("aise",         "High income",    "#FF5722", rasters_income["aise"]),
]

socio_vars = {
    "freq_crim_norm" : "Crime frequency (mean)",
    "desserte_score" : "Transit accessibility score",
    "precarity_norm" : "Precarity index (normalised)",
}

for var_col, var_label in socio_vars.items():

    fig, ax = plt.subplots(figsize=(10, 6))
    # fig.suptitle(
    #     f"Pedestrian density vs {var_label} — by group\nGIREC zones",
    #     fontsize=13, fontweight="bold"
    # )

    for group_name, group_label, color, _ in groups_to_plot:  # ← _ pour ignorer le raster

        density_col = f'density_{group_name}'

        if density_col not in zones_girec.columns:
            print(f"⚠ {density_col} absent de zones_girec — skippé")
            continue

        df_plot = pd.DataFrame({
            "density" : zones_girec[density_col].values,
            var_col   : zones_girec[var_col].values
        }).dropna()
        df_plot = df_plot[df_plot["density"] > 0]

        if len(df_plot) < 10:
            continue

        x = df_plot[var_col].values
        y = df_plot["density"].values

        r, p = spearmanr(x, y)
        sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

        log_y       = np.log1p(y)
        lowess_vals = lowess(log_y, x, frac=0.4)
        ax.plot(lowess_vals[:, 0], np.expm1(lowess_vals[:, 1]),
                color=color, linewidth=2,
                label=f"{group_label} (r={r:.4f}{sig})")

    ax.set_xlabel(var_label, fontsize=12)
    ax.set_ylabel("Mean daily pedestrian density\n(legs · day⁻¹ · user⁻¹)", fontsize=12)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Groupes à comparer ───────────────────────────────────────────────────────
groups_to_plot = [
    # Gender
    ("homme",        "Men",                  ALL_GROUP_COLORS["homme"],        rasters_gender["homme"]),
    ("femme",        "Women",                ALL_GROUP_COLORS["femme"],        rasters_gender["femme"]),
    # Age
    ("18-29",        "18-29",                ALL_GROUP_COLORS["18-29"],        rasters_age["18-29"]),
    ("30-44",        "30-44",                ALL_GROUP_COLORS["30-44"],        rasters_age["30-44"]),
    ("45-59",        "45-59",                ALL_GROUP_COLORS["45-59"],        rasters_age["45-59"]),
    ("60+",          "60+",                  ALL_GROUP_COLORS["60+"],          rasters_age["60+"]),
    # Income
    ("tres_modeste", "Low income",           ALL_GROUP_COLORS["tres_modeste"], rasters_income["tres_modeste"]),
    ("modeste",      "Median income",        ALL_GROUP_COLORS["modeste"],      rasters_income["modeste"]),
    ("aise",         "High income",          ALL_GROUP_COLORS["aise"],         rasters_income["aise"]),
    # Car
    ("no_car",       "No car",               ALL_GROUP_COLORS["no_car"],       rasters_car["no_car"]),
    ("has_car",      "Car owner",            ALL_GROUP_COLORS["has_car"],      rasters_car["has_car"]),
    # PT
    ("no_tp",        "No PT sub.",           ALL_GROUP_COLORS["no_tp"],        rasters_tp["no_tp"]),
    ("partial_tp",   "Partial PT sub.",      ALL_GROUP_COLORS["partial_tp"],   rasters_tp["partial_tp"]),
    ("full_tp",      "Full PT sub.",         ALL_GROUP_COLORS["full_tp"],      rasters_tp["full_tp"]),
]

socio_vars = {
    #"freq_crim_norm" : "Crime frequency (mean)",
    "desserte_score" : "Transit accessibility score",
    #"precarity_norm" : "Precarity index (normalised)",
}

for var_col, var_label in socio_vars.items():

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    print(f"\n{'='*60}")
    print(f"Spearman correlations — {var_label}")
    print(f"{'='*60}")
    print(f"{'Group':<25} {'r':>8} {'p-value':>12} {'Sig':>6}")
    print(f"{'─'*55}")

    for group_name, group_label, color, _ in groups_to_plot:

        density_col = f'density_{group_name}'

        if density_col not in zones_girec.columns:
            print(f"⚠ {density_col} absent de zones_girec — skippé")
            continue

        df_plot = pd.DataFrame({
            "density" : zones_girec[density_col].values,
            var_col   : zones_girec[var_col].values
        }).dropna()
        df_plot = df_plot[df_plot["density"] > 0]

        if len(df_plot) < 10:
            continue

        x = df_plot[var_col].values
        y = df_plot["density"].values

        r, p = spearmanr(x, y)
        sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

        # ─── Print des coefficients ───────────────────────────────────────────
        print(f"{group_label:<25} {r:>8.4f} {p:>12.2e} {sig:>6}")

        log_y       = np.log1p(y)
        lowess_vals = lowess(log_y, x, frac=0.4)
        ax.plot(lowess_vals[:, 0], np.expm1(lowess_vals[:, 1]),
                color=color, linewidth=2,
                label=group_label)  # ← sans le score dans le label

    ax.set_xlabel(var_label, fontsize=12)
    ax.set_ylabel("Mean daily pedestrian density \n (legs · day⁻¹ · user⁻¹)", fontsize=12)
    ax.legend(fontsize=10, loc="upper center",
              bbox_to_anchor=(0.5, -0.18),
              ncol=6, frameon=False)   # ← légende sous le graphique, 4 colonnes
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"{'='*60}")

In [ ]:
density_cols_to_test = [c for c in zones_girec.columns if c.startswith('density_')]

print(f"{'Groupe':<30} {'r':>8} {'p':>10} {'sig':>6}")
print(f"{'-'*56}")

results = []
for col in density_cols_to_test:
    mask = zones_girec[[col, 'precarite_score_24']].notna().all(axis=1)
    mask = mask & (zones_girec[col] > 0)
    
    if mask.sum() < 10:
        continue
    
    r, p = spearmanr(
        zones_girec.loc[mask, col],
        zones_girec.loc[mask, 'precarite_score_24']
    )
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    group = col.replace('density_', '')
    results.append({'group': group, 'r': r, 'p': p, 'sig': sig})
    print(f"  {group:<28} {r:>8.3f} {p:>10.4f} {sig:>6}")

In [ ]:
# Ratio densité dans zones précaires vs densité globale par groupe
for col in ['density_tres_modeste', 'density_modeste', 'density_aise']:
    mean_precarious = zones_girec[zones_girec['precarite_score_24'] >= 5][col].mean() #densité moyenne de passage du groupe uniquement dans les zones très précaires (>= 5)
    mean_all        = zones_girec[col].mean() #densité moyenne de groupe sur toutes les zones du canton
    ratio           = mean_precarious / mean_all # rapport entre les deux 
    print(f"{col:<30} ratio précaire/global : {ratio:.3f}")

- ratio = 1.0 → le groupe marche autant dans les zones précaires que partout ailleurs — pas de surreprésentation
- ratio > 1.0 → le groupe marche plus dans les zones précaires que sa moyenne globale → surreprésenté
- ratio < 1.0 → le groupe marche moins dans les zones précaires → sous-représenté

# CARREAUX 200

In [ ]:
# ─── Scatter plots densité vs variables socio-spatiales par groupe ────────────
from scipy.stats import spearmanr
from statsmodels.nonparametric.smoothers_lowess import lowess

socio_vars = {
    #"freq_crim_mean" : "Crime frequency (mean)",
    "desserte_score" : "Transit accessibility score",
    "precarity_norm" : "Precarity index (normalised)",
}

# ─── Groupes à comparer ───────────────────────────────────────────────────────
groups_to_plot = [
    ("homme",      "Men",          "#4C72B0", rasters_gender["homme"]),
    ("femme",      "Women",        "#DD8452", rasters_gender["femme"]),
    ("18-29",      "18-29",        "#55A868", rasters_age["18-29"]),
    ("60+",        "60+",          "#C44E52", rasters_age["60+"]),
    ("no_car",     "No car",       "#8172B2", rasters_car["no_car"]),
    ("has_car",    "Car owner",    "#937860", rasters_car["has_car"]),
    ("no_tp",      "No PT sub.",   "#DA8BC3", rasters_tp["no_tp"]),
    ("full_tp",    "Full PT sub.", "#8C8C8C", rasters_tp["full_tp"]),
    ("tres_modeste", "Low income",     "#2196F3", rasters_income["tres_modeste"]),
    ("aise",         "High income",    "#FF5722", rasters_income["aise"]),
]

for var_col, var_label in socio_vars.items():

    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        f"Pedestrian density vs {var_label} — by group\nCarreau 200x200m",
        fontsize=13, fontweight="bold"
    )

    for group_name, group_label, color, raster in groups_to_plot:

        # ─── Ajouter density du groupe aux carreaux ──────────────────────────
        # On utilise rasterstats pour agréger le raster sur les zones
        import rasterstats
        raster_filled = np.where(np.isnan(raster), -9999, raster)
        stats = rasterstats.zonal_stats(
            carreau_200.geometry,
            raster_filled,
            affine=transform,
            stats=["mean"],
            nodata=-9999
        )
        density_vals = np.array([s["mean"] if s["mean"] is not None else np.nan for s in stats])

        # ─── DataFrame pour ce groupe ─────────────────────────────────────────
        df_plot = pd.DataFrame({
            "density" : density_vals,
            var_col   : carreau_200[var_col].values
        }).dropna()
        df_plot = df_plot[df_plot["density"] > 0]

        if len(df_plot) < 10:
            continue

        x = df_plot[var_col].values
        y = df_plot["density"].values

        # ─── Corrélation ──────────────────────────────────────────────────────
        r, p = spearmanr(x, y)
        sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

        # ─── LOWESS uniquement (pas de scatter pour ne pas surcharger) ────────
        log_y       = np.log1p(y)
        lowess_vals = lowess(log_y, x, frac=0.4)
        ax.plot(lowess_vals[:, 0], np.expm1(lowess_vals[:, 1]),
                color=color, linewidth=2,
                label=f"{group_label} (r={r:.2f}{sig})")

    #ax.set_yscale('log')
    ax.set_xlabel(var_label, fontsize=11)
    ax.set_ylabel("Mean daily pedestrian density\n(legs · day⁻¹ · user⁻¹)", fontsize=10)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

### Comparaison distribution score precarite GIREC vs Carreau_200

In [ ]:
# Comparer les distributions
print("zones_girec precarite_score_24 :")
print(zones_girec['precarite_score_24'].value_counts().sort_index())

print("\nagglo_carreau precarite_score_24 :")
print(carreau_200['precarite_score_24'].value_counts().sort_index())

In [ ]:
print("zones_girec precarite_score_24 :")
print(zones_girec['precarite_score_24'].describe())

print("\nagglo_carreau precarite_score_24 :")
print(carreau_200['precarite_score_24'].describe())

In [ ]:
print("zones_girec precarity_norm :")
print(zones_girec['precarity_norm'].describe().round(4))

print("\ncarreau_200 precarity_norm :")
print(carreau_200['precarity_norm'].describe().round(4))

# Et surtout, est-ce que les distributions sont cohérentes ?
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
zones_girec['precarity_norm'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('precarity_norm — GIREC')
carreau_200['precarity_norm'].hist(bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('precarity_norm — Carreau 200m')
plt.tight_layout()
plt.show()

In [ ]:
print("zones_girec precarite_score_24 :")
print(zones_girec['precarite_score_24'].value_counts().sort_index())

In [ ]:
print("\ncarreau_200 precarite_score_24 :")
print(carreau_200['precarite_score_24'].value_counts().sort_index())

In [ ]:
# Recalculer les deux normalisations et comparer
girec_min = zones_girec['precarite_score_24'].min()
girec_max = zones_girec['precarite_score_24'].max()

carreau_min = carreau_200['precarite_score_24'].min()
carreau_max = carreau_200['precarite_score_24'].max()

print(f"zones_girec  : min={girec_min} | max={girec_max}")
print(f"carreau_200  : min={carreau_min} | max={carreau_max}")

In [ ]:
# Recalcul propre avec la même formule
zones_girec['precarity_norm']  = zones_girec['precarite_score_24'] / 6
carreau_200['precarity_norm']  = carreau_200['precarite_score_24'] / 6

print("zones_girec precarity_norm recalculé :")
print(zones_girec['precarity_norm'].describe().round(4))

print("\ncarreau_200 precarity_norm recalculé :")
print(carreau_200['precarity_norm'].describe().round(4))

# AGGREGATION DENSITY - CARREAU_200

In [ ]:
# ─── Enrichir carreau_200 avec toutes les dimensions ─────────────────────────
print("── Agrégation All users ──────────────────────────────")
aggregate_raster_to_area(carreau_200, {"all": raster_mean_clipped_v2}, [("all", None)], transform)

print("\n── Agrégation Gender ─────────────────────────────────")
aggregate_raster_to_area(carreau_200, rasters_gender, gender_filters, transform)

print("\n── Agrégation Age ────────────────────────────────────")
aggregate_raster_to_area(carreau_200, rasters_age, age_filters, transform)

print("\n── Agrégation Income ─────────────────────────────────")
aggregate_raster_to_area(carreau_200, rasters_income, income_filters, transform)

print("\n── Agrégation Car ownership ──────────────────────────")
aggregate_raster_to_area(carreau_200, rasters_car, car_filters, transform)

print("\n── Agrégation PT subscriber ──────────────────────────")
aggregate_raster_to_area(carreau_200, rasters_tp, tp_filters, transform)

# ─── Vérification ─────────────────────────────────────────────────────────────
density_cols = [c for c in carreau_200.columns if c.startswith("density_")]
print(f"\n✓ Colonnes density ajoutées : {density_cols}")
print(f"✓ Carreaux total            : {len(carreau_200)}")
print(carreau_200[density_cols].describe().round(6))

# RASTER DIFFERENCE

In [ ]:
# ─── Comparaisons à tester ────────────────────────────────────────────────────
comparisons = [
    # (raster_dict, group_A, group_B, label)
    (rasters_age,    "18-29",        "60+",         "Age: 18-29 vs 60+"),
    (rasters_age,    "30-44",        "45-59",       "Age: 30-44 vs 45-59"),
    (rasters_gender, "homme",        "femme",       "Gender: Male vs Female"),
    (rasters_income, "tres_modeste", "aise",        "Income: Low vs High"),
    (rasters_car,    "no_car",       "has_car",         "Car: No car vs Car"),
]

percentiles_range = list(range(90, 100))

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(comparisons), 3, figsize=(15, 4 * len(comparisons)))
fig.suptitle("Sensitivity analysis — min_intensity_percentile | no threshold", 
             fontsize=13, fontweight='bold')

for i, (rasters_dict, group_a, group_b, label) in enumerate(comparisons):
    rows = []
    for p in percentiles_range:
        ndpdi_test = compute_ndpdi(
            raster_a                 = rasters_dict[group_a],
            raster_b                 = rasters_dict[group_b],
            canton_mask              = canton_GE_mask,
            group_a_label            = labels_all_groups[group_a],
            group_b_label            = labels_all_groups[group_b],
            threshold                = None,
            min_intensity_brut       = None,
            min_intensity_percentile = p
        )
        valid = ndpdi_test[~np.isnan(ndpdi_test)]
        rows.append({
            "percentile"     : p,
            "pixels_valides" : len(valid),
            "pct_canton"     : len(valid) / canton_GE_mask.sum() * 100,
            "mean_NDPDI"     : round(valid.mean(), 3) if len(valid) > 0 else np.nan,
            "pct_A_gt_B"     : round((valid > 0).mean() * 100, 1) if len(valid) > 0 else np.nan,
        })

    df_sens = pd.DataFrame(rows)

    # Pixels valides
    axes[i, 0].plot(df_sens["percentile"], df_sens["pct_canton"], 
                    marker='o', color='steelblue')
    axes[i, 0].set_ylabel("% canton covered")
    axes[i, 0].set_title(f"{label}\nSpatial coverage")
    axes[i, 0].axvline(97, color='red', linestyle='--', alpha=0.7, label='P97')
    axes[i, 0].legend(fontsize=8)

    # Mean NDPDI
    axes[i, 1].plot(df_sens["percentile"], df_sens["mean_NDPDI"], 
                    marker='o', color='darkorange')
    axes[i, 1].set_ylabel("Mean NDPDI")
    axes[i, 1].set_title(f"{label}\nMean NDPDI")
    axes[i, 1].axvline(97, color='red', linestyle='--', alpha=0.7, label='P97')
    axes[i, 1].legend(fontsize=8)

    # % A > B
    axes[i, 2].plot(df_sens["percentile"], df_sens["pct_A_gt_B"], 
                    marker='o', color='seagreen')
    axes[i, 2].set_ylabel("% pixels A > B")
    axes[i, 2].set_title(f"{label}\n% A > B")
    axes[i, 2].axvline(97, color='red', linestyle='--', alpha=0.7, label='P97')
    axes[i, 2].legend(fontsize=8)

    # xlabel seulement sur la dernière ligne
    if i == len(comparisons) - 1:
        for j in range(3):
            axes[i, j].set_xlabel("min_intensity_percentile")

plt.tight_layout()
#plt.savefig("sensitivity_ndpdi_all_comparisons.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_p, diag = find_balanced_percentile(rasters_gender["homme"], rasters_gender["femme"])

In [ ]:
best_p, diag = find_balanced_percentile(rasters_car["no_car"], rasters_car["has_car"])

In [ ]:
best_p, diag = find_balanced_percentile(rasters_age["18-29"], rasters_age["60+"])

In [ ]:
best_p, diag = find_balanced_percentile(rasters_income["tres_modeste"], rasters_income["aise"])

In [ ]:
raster_A = "homme"
raster_B = "femme"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_income = compute_ndpdi(
    raster_a      = rasters_gender[raster_A],
    raster_b      = rasters_gender[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile= 0,
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask
)

In [ ]:
raster_A = "homme"
raster_B = "femme"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_income = compute_ndpdi(
    raster_a      = rasters_gender[raster_A],
    raster_b      = rasters_gender[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile= 95,
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

landmarks_gender = {k: LANDMARKS_GE[k] for k in ["Lancy-Pont-Rouge \n Train Station", "Geneva University \n Hospitals (HUG)"]}
print("zoom_bounds actuel :", xmin_z, ymin_z, xmax_z, ymax_z)
# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask,
    landmarks=landmarks_gender
)

In [ ]:
raster_A = "18-29"
raster_B = "60+"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_income = compute_ndpdi(
    raster_a      = rasters_age[raster_A],
    raster_b      = rasters_age[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile= 0,
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

print("zoom_bounds actuel :", xmin_z, ymin_z, xmax_z, ymax_z)
# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask
)

In [ ]:
raster_A = "18-29"
raster_B = "60+"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_income = compute_ndpdi(
    raster_a      = rasters_age[raster_A],
    raster_b      = rasters_age[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile= 95,
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)
landmarks_age = {k: LANDMARKS_GE[k] for k in [
    "Lancy-Pont-Rouge \n Train Station",
    "Geneva Cornavin \n Train Station",
    "Airport",
    "Plainpalais",
    "Sciences University",
    "Geneva University",
    "Human Science University \n (Uni Mai /Uni Pignon)",
    "HEPIA",
    "HEAD",
    "United Nation \n (ONU)",
    "WTO (OMC)"
]}
print("zoom_bounds actuel :", xmin_z, ymin_z, xmax_z, ymax_z)
# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask,
    landmarks=landmarks_age
)

In [ ]:
raster_A = "tres_modeste"
raster_B = "aise"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_income = compute_ndpdi(
    raster_a      = rasters_income[raster_A],
    raster_b      = rasters_income[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile= 0,
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

print("zoom_bounds actuel :", xmin_z, ymin_z, xmax_z, ymax_z)
# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask
)

In [ ]:
raster_A = "tres_modeste"
raster_B = "aise"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_income = compute_ndpdi(
    raster_a      = rasters_income[raster_A],
    raster_b      = rasters_income[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile= 95,
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

landmarks_income = {k: LANDMARKS_GE[k] for k in [
    "Lancy-Pont-Rouge \n Train Station",
    "Geneva Cornavin \n Train Station",
    "Pictet \n (Wealth Management)",
    "Rhône Street"
]}
print("zoom_bounds actuel :", xmin_z, ymin_z, xmax_z, ymax_z)
# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = "", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask,
    landmarks=landmarks_income
)

In [ ]:
raster_A = "no_car"
raster_B = "has_car"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_income = compute_ndpdi(
    raster_a      = rasters_car[raster_A],
    raster_b      = rasters_car[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile = 0
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_income,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask
)

In [ ]:
raster_A = "no_car"
raster_B = "has_car"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_car = compute_ndpdi(
    raster_a      = rasters_car[raster_A],
    raster_b      = rasters_car[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile = 95
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_car,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_car,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask
)

In [ ]:
raster_A = "no_tp"
raster_B = "full_tp"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_tp = compute_ndpdi(
    raster_a      = rasters_tp[raster_A],
    raster_b      = rasters_tp[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile = 0
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_tp,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_tp,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask
)

In [ ]:
raster_A = "no_tp"
raster_B = "full_tp"

# ─── Calcul NDPDI ─────────────────────────────────────────────────────────────
ndpdi_tp = compute_ndpdi(
    raster_a      = rasters_tp[raster_A],
    raster_b      = rasters_tp[raster_B],
    canton_mask   = canton_GE_mask,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    threshold     = 0,
    min_intensity_brut = None,
    min_intensity_percentile = 95
)

# ─── Canton entier ────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_tp,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    canton_mask   = canton_GE_mask   
)

landmarks_tp = {k: LANDMARKS_GE[k] for k in [
    "Geneva Cornavin \n Train Station",
    "Lancy-Pont-Rouge \n Train Station",
    "Geneva Champelle \n Train Station",
]}

# ─── Zoom commune ─────────────────────────────────────────────────────────────
plot_ndpdi(
    ndpdi         = ndpdi_tp,
    title         = f"", #f"NDPDI — {labels_all_groups[raster_A]} vs {labels_all_groups[raster_B]} | Zoom {focus_commune_name}",
    extent        = extent,
    canton_GE     = canton_GE,
    girec         = girec,
    group_a_label = labels_all_groups[raster_A],
    group_b_label = labels_all_groups[raster_B],
    focus         = focus_commune,
    zoom_bounds   = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin        = margin,
    canton_mask   = canton_GE_mask,
    landmarks=landmarks_tp
)

In [ ]:
percentiles = [90, 95, 97, 99]

rows = []
for p in percentiles:
    ndpdi_test = compute_ndpdi(
        raster_a                 = rasters_age["18-29"],
        raster_b                 = rasters_age["60+"],
        canton_mask              = canton_GE_mask,
        group_a_label            = labels_all_groups["18-29"],
        group_b_label            = labels_all_groups["60+"],
        threshold                = None,   # ← pas de threshold
        min_intensity_brut       = None,
        min_intensity_percentile = p
    )
    valid = ndpdi_test[~np.isnan(ndpdi_test)]
    rows.append({
        "percentile"     : f"P{p}",
        "pixels_valides" : len(valid),
        "mean_NDPDI"     : round(valid.mean(), 3) if len(valid) > 0 else np.nan,
        "pct_A_gt_B"     : round((valid > 0).mean() * 100, 1) if len(valid) > 0 else np.nan,
        "pct_B_gt_A"     : round((valid < 0).mean() * 100, 1) if len(valid) > 0 else np.nan,
    })

df_sensitivity = pd.DataFrame(rows)
print(df_sensitivity.to_string(index=False))

# DENSITY VS WALK_INDEX

## ALL

In [ ]:
# ─── Construction dynamique depuis labels_all_groups ─────────────────────────
rasters_all_groups = {
    "all"          : raster_mean_clipped_v2,
    **rasters_gender,
    **rasters_age,
    **rasters_income,
    **rasters_car,
    **rasters_tp,
}

dimensions = {
    "all"          : "All users",
    "homme"        : "Gender",
    "femme"        : "Gender",
    "18-29"        : "Age",
    "30-44"        : "Age",
    "45-59"        : "Age",
    "60+"          : "Age",
    "tres_modeste" : "Income",
    "modeste"      : "Income",
    "median"       : "Income",
    "aise"         : "Income",
    "no_car"       : "Car ownership",
    "has_car"      : "Car ownership",
    "no_tp"        : "PT subscription",
    "partial_tp"   : "PT subscription",
    "full_tp"      : "PT subscription",
}

# ─── groups_to_analyse construit automatiquement ─────────────────────────────
groups_to_analyse = [
    (key, labels_all_groups[key], rasters_all_groups[key], dimensions[key])
    for key in labels_all_groups
    if key in rasters_all_groups
]

# ─── Corrélation Spearman density vs walk_index par groupe ───────────────────
spearman_results_girec_density_walk = {}

for key, label in labels_all_groups.items():
    col = f"density_{key}"
    
    if col not in zones_girec.columns:
        print(f"  ⚠ {col} manquant dans zones_girec — skippé")
        continue

    df = zones_girec[[col, "walk_index"]].dropna()
    df = df[df[col] > 0]

    if len(df) < 10:
        continue

    r, p = spearmanr(df[col], df["walk_index"])
    sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

    spearman_results_girec_density_walk[key] = {
        "label"    : label,
        "dimension": dimensions[key],
        "r"        : r,
        "p"        : p,
        "sig"      : sig,
        "n"        : len(df),
        "area_%": len(df)/len(zones_girec),
    }
    print(f"  {label:<25} r={r:.3f} {sig} | n={len(df)} | area % ={len(df)/len(zones_girec):.3f}")

print(f"\n✓ {len(spearman_results_girec_density_walk)} groupes analysés")


In [ ]:
# ─── Corrélation Spearman density vs walk_index par groupe ───────────────────
spearman_results_carreau_200_density_walk = {}

for key, label in labels_all_groups.items():
    col = f"density_{key}"
    
    if col not in carreau_200.columns:
        print(f"  ⚠ {col} manquant dans zones_girec — skippé")
        continue

    df = carreau_200[[col, "walk_index"]].dropna()
    df = df[df[col] > 0]

    if len(df) < 10:
        continue

    r, p = spearmanr(df[col], df["walk_index"])
    sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

    spearman_results_carreau_200_density_walk[key] = {
        "label"    : label,
        "dimension": dimensions[key],
        "r"        : r,
        "p"        : p,
        "sig"      : sig,
        "n"        : len(df),
        "area_%"   : len(df) / len(carreau_200),
    }
    print(f"  {label:<25} r={r:.3f} {sig} | n={len(df)} | area % ={len(df)/len(carreau_200):.3f}")

print(f"\n✓ {len(spearman_results_carreau_200_density_walk)} groupes analysés")

In [ ]:
# ─── Affichage tableau des résultats ─────────────────────────────────────────
df_spearman_girec = pd.DataFrame([
    {
        "Group"     : spearman_results_girec_density_walk[k]["label"],
        "Dimension" : spearman_results_girec_density_walk[k]["dimension"],
        "r"         : round(spearman_results_girec_density_walk[k]["r"], 4),
        "p-value"   : f"{spearman_results_girec_density_walk[k]['p']:.2e}",
        "Sig"       : spearman_results_girec_density_walk[k]["sig"],
        "n zones"   : spearman_results_girec_density_walk[k]["n"],
        "area %"    : f"{spearman_results_girec_density_walk[k]['area_%']:.2f}",
    }
    for k in spearman_results_girec_density_walk
])

display(df_spearman_girec.style
    .background_gradient(subset=["r"], cmap="RdYlGn", vmin=-1, vmax=1)
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]}
    ])
)

In [ ]:
# ─── Affichage tableau des résultats ─────────────────────────────────────────
df_spearman_carreau_200 = pd.DataFrame([
    {
        "Group"     : spearman_results_carreau_200_density_walk[k]["label"],
        "Dimension" : spearman_results_carreau_200_density_walk[k]["dimension"],
        "r"         : round(spearman_results_carreau_200_density_walk[k]["r"], 4),
        "p-value"   : f"{spearman_results_carreau_200_density_walk[k]['p']:.2e}",
        "Sig"       : spearman_results_carreau_200_density_walk[k]["sig"],
        "n zones"   : spearman_results_carreau_200_density_walk[k]["n"],
        "area %"    : f"{spearman_results_carreau_200_density_walk[k]['area_%']:.2f}",
    }
    for k in spearman_results_carreau_200_density_walk
])

display(df_spearman_carreau_200.style
    .background_gradient(subset=["r"], cmap="RdYlGn", vmin=-1, vmax=1)
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]}
    ])
)

In [ ]:
# ─── Barplot Spearman density vs walk_index par groupe ───────────────────────
dim_colors = {
    "All users"       : "#aaaaaa",
    "Gender"          : "#4C72B0",
    "Age"             : "#55A868",
    "Income"          : "#C44E52",
    "Car ownership"   : "#DD8452",
    "PT subscription" : "#8172B2",
}

keys   = list(spearman_results_girec_density_walk.keys())
labels = [spearman_results_girec_density_walk[k]["label"]     for k in keys]
r_vals = [spearman_results_girec_density_walk[k]["r"]         for k in keys]
sigs   = [spearman_results_girec_density_walk[k]["sig"]       for k in keys]
colors = [dim_colors.get(spearman_results_girec_density_walk[k]["dimension"], "#8C8C8C") for k in keys]

x = np.arange(len(keys))

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle("Spearman correlation — Pedestrian density vs Walk index by group\n(GIREC zones)",
             fontsize=13, fontweight="bold")

bars = ax.bar(x, r_vals, color=colors, edgecolor="white", linewidth=0.5)

# ─── Valeurs + significativité au-dessus des barres ──────────────────────────
for bar, val, sig in zip(bars, r_vals, sigs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.3f}\n{sig}",
            ha='center', va='bottom', fontsize=7)

# ─── Ligne de référence à 0 ───────────────────────────────────────────────────
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

# ─── Séparateurs entre dimensions ────────────────────────────────────────────
prev_dim = None
for i, k in enumerate(keys):
    dim = spearman_results_girec_density_walk[k]["dimension"]
    if dim != prev_dim and i > 0:
        ax.axvline(i - 0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
    prev_dim = dim

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel("Spearman r")
ax.set_ylim(-0.1, max(r_vals) + 0.15)
ax.grid(axis='y', alpha=0.3)

# ─── Légende dimensions ───────────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=color, label=dim)
    for dim, color in dim_colors.items()
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8,
          title="Dimension", title_fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Barplot Spearman density vs walk_index par groupe ───────────────────────


keys   = list(spearman_results_carreau_200_density_walk.keys())
labels = [spearman_results_carreau_200_density_walk[k]["label"]     for k in keys]
r_vals = [spearman_results_carreau_200_density_walk[k]["r"]         for k in keys]
sigs   = [spearman_results_carreau_200_density_walk[k]["sig"]       for k in keys]
colors = [dim_colors.get(spearman_results_carreau_200_density_walk[k]["dimension"], "#8C8C8C") for k in keys]

x = np.arange(len(keys))

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle("Spearman correlation — Pedestrian density vs Walk index by group\n(200x200m zones)",
             fontsize=13, fontweight="bold")

bars = ax.bar(x, r_vals, color=colors, edgecolor="white", linewidth=0.5)

# ─── Valeurs + significativité au-dessus des barres ──────────────────────────
for bar, val, sig in zip(bars, r_vals, sigs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.3f}\n{sig}",
            ha='center', va='bottom', fontsize=7)

# ─── Ligne de référence à 0 ───────────────────────────────────────────────────
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

# ─── Séparateurs entre dimensions ────────────────────────────────────────────
prev_dim = None
for i, k in enumerate(keys):
    dim = spearman_results_carreau_200_density_walk[k]["dimension"]
    if dim != prev_dim and i > 0:
        ax.axvline(i - 0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
    prev_dim = dim

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel("Spearman r")
ax.set_ylim(-0.1, max(r_vals) + 0.15)
ax.grid(axis='y', alpha=0.3)

# ─── Légende dimensions ───────────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=color, label=dim)
    for dim, color in dim_colors.items()
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8,
          title="Dimension", title_fontsize=9)

plt.tight_layout()
plt.show()

## DENSITY VS SPECIFIC WALK INDEX

### WOMEN

#### GIREC

In [ ]:
# Femmes — GIREC
women_comparison_results_GIREC = compute_spearman_comparison(
    gdf                = zones_girec,
    density_col        = "density_femme",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_women",
    group_label        = "Women",
)

In [ ]:
plot_spearman_comparison(
    results            = women_comparison_results_GIREC,
    scale_label        = "GIREC zones",
    group_label        = "Women",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_women",
)

In [ ]:
women_density_walk_linear_comparison_GIREC = plot_linear_comparison(
    gdf                = zones_girec,
    density_col        = "density_femme",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_women",
    group_label        = "Women",
    scale_label        = "GIREC zones",
)

#### CARREAU_200

In [ ]:
# Femmes — carreau 200m
women_comparison_results_carreau_200 = compute_spearman_comparison(
    gdf                = carreau_200,
    density_col        = "density_femme",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_women",
    group_label        = "Women",
)

In [ ]:
plot_spearman_comparison(
    results            = women_comparison_results_carreau_200,
    scale_label        = "200x200m grid",
    group_label        = "Women",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_women",
)

In [ ]:
women_density_walk_linar_comparison_carreau_200 = plot_linear_comparison(
    gdf                = carreau_200,
    density_col        = "density_femme",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_women",
    group_label        = "Women",
    scale_label        = "200x200m grid",
)

### SENIOR

#### GIREC

In [ ]:
# Senior — GIREC
senior_comparison_results_GIREC = compute_spearman_comparison(
    gdf                = zones_girec,
    density_col        = "density_60+",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_senior",
    group_label        = "60+",
)

In [ ]:
plot_spearman_comparison(
    results            = senior_comparison_results_GIREC,
    scale_label        = "GIREC zones",
    group_label        = "60+",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_senior",
)

In [ ]:
senior_density_walk_linear_comparison_GIREC = plot_linear_comparison(
    gdf                = zones_girec,
    density_col        = "density_60+",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_senior",
    group_label        = "60+",
    scale_label        = "GIREC zones",
)

In [ ]:
# Senior — carreau_200
senior_comparison_results_carreau_200 = compute_spearman_comparison(
    gdf                = carreau_200,
    density_col        = "density_60+",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_senior",
    group_label        = "60+",
)

In [ ]:
plot_spearman_comparison(
    results            = senior_comparison_results_carreau_200,
    scale_label        = "200x200m grid",
    group_label        = "60+",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_senior",
)

In [ ]:
senior_density_walk_linear_comparison_carreau_200 = plot_linear_comparison(
    gdf                = carreau_200,
    density_col        = "density_60+",
    index_col_standard = "walk_index",
    index_col_specific = "walk_index_senior",
    group_label        = "60+",
    scale_label        = "200x200m grid",
)

In [ ]:
# ─── Scatter + LOWESS — Men vs Women ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle("Pedestrian density vs Walk index\nMen vs Women — GIREC zones",
             fontsize=13, fontweight="bold")

for key, color in [
    ("homme", "#4C72B0"),
    ("femme", "#DD8452"),
]:
    col   = f"density_{key}"
    label = labels_all_groups[key]
    r     = spearman_results_girec_density_walk[key]["r"]
    sig   = spearman_results_girec_density_walk[key]["sig"]

    df = zones_girec[[col, "walk_index"]].dropna()
    df = df[df[col] > 0]

    x = df["walk_index"].values
    y = df[col].values

    # ─── Scatter ──────────────────────────────────────────────────────────────
    ax.scatter(x, y, alpha=0.2, s=10, color=color)

    # ─── LOWESS ───────────────────────────────────────────────────────────────
    log_y       = np.log1p(y)
    lowess_vals = lowess(log_y, x, frac=0.4)
    ax.plot(lowess_vals[:, 0], np.expm1(lowess_vals[:, 1]),
            color=color, linewidth=2.5,
            label=f"{label} (r={r:.2f}{sig})")

ax.set_yscale('log')
ax.set_xlabel("Walk index", fontsize=11)
ax.set_ylabel("Mean daily pedestrian density\n(legs · day⁻¹ · user⁻¹) — log scale", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Scatter + LOWESS — No car vs Car owner ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle("Pedestrian density vs Walk index\nNo car vs Car owner — GIREC zones",
             fontsize=13, fontweight="bold")

for key, color in [
    ("no_car",  "#8172B2"),
    ("has_car", "#937860"),
]:
    col   = f"density_{key}"
    label = labels_all_groups[key]

    df = zones_girec[[col, "walk_index"]].dropna()
    df = df[df[col] > 0]

    x = df["walk_index"].values
    y = df[col].values

    # ─── Scatter ──────────────────────────────────────────────────────────────
    ax.scatter(x, y, alpha=0.2, s=10, color=color)

    # ─── LOWESS ───────────────────────────────────────────────────────────────
    log_y       = np.log1p(y)
    lowess_vals = lowess(log_y, x, frac=0.4)
    ax.plot(lowess_vals[:, 0], np.expm1(lowess_vals[:, 1]),
            color=color, linewidth=2.5, label=label)

ax.set_yscale('log')
ax.set_xlabel("Walk index", fontsize=11)
ax.set_ylabel("Mean daily pedestrian density\n(legs · day⁻¹ · user⁻¹) — log scale", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plot_delta_map('women',  zones_girec,   spatial_level='girec')
plot_delta_map('women',  carreau_200, spatial_level='carreau')
plot_delta_map('senior', zones_girec,   spatial_level='girec')
plot_delta_map('senior', carreau_200, spatial_level='carreau')

# WALK-INDEX VS LOCATION

In [ ]:
print(zones_girec.dtypes.to_string())

In [ ]:
# ─── Gradient de couleur automatique ─────────────────────────────────────────
cmap_area  = cm.get_cmap("RdYlBu", len(AREA_TYPE_ORDER))
area_colors = {
    area_type: cmap_area(i / (len(AREA_TYPE_ORDER) - 1))
    for i, area_type in enumerate(AREA_TYPE_ORDER)
}

fig, ax = plt.subplots(figsize=(10, 10))
#fig.suptitle("GIREC and Municipalities zones by AREA_TYPE", fontsize=13, fontweight="bold")
fig.patch.set_alpha(0)   # ← fond de la figure transparent
ax.patch.set_alpha(0) 

zones_girec.plot(ax=ax, color="#e0e0e0", linewidth=0.3, edgecolor="white")

for area_type in AREA_TYPE_ORDER:
    subset = zones_girec[zones_girec["AREA_TYPE"] == area_type]
    if len(subset) > 0:
        subset.plot(ax=ax, color=AREA_TYPE_COLORS[area_type],
                    linewidth=0.3, edgecolor="white")

canton_GE.boundary.plot(ax=ax, color='black', linewidth=1.5)
ax.set_axis_off()

# ── Délimitation des communes ─────────────────────────────────────────────
zones_communes_GE_fusionnee.boundary.plot(ax=ax, color='black', linewidth=1.0)

# ── Noms des communes ─────────────────────────────────────────────────────
dark_communes = {"Genève", "Carouge", "Lancy"} 

for _, row in zones_communes_GE_fusionnee.iterrows():
    centroid = row.geometry.centroid
    text_color = "black" if row["COMMUNE"] in dark_communes else "black"
    
    ax.annotate(
        row['COMMUNE'],
        xy=(centroid.x, centroid.y),
        fontsize=6,
        ha='center',
        va='center',
        color=text_color,
        fontweight='bold'
    )


canton_GE.boundary.plot(ax=ax, color='black', linewidth=1)
ax.set_axis_off()

# ─── Légende manuelle ─────────────────────────────────────────────────────────
legend_handles = [
    Patch(facecolor=AREA_TYPE_COLORS[area_type], edgecolor="gray",
          label=AREA_TYPE_LABELS.get(area_type, area_type))
    for area_type in AREA_TYPE_ORDER
    if area_type in zones_girec["AREA_TYPE"].values
]

ax.legend(handles=legend_handles, fontsize=10,
          frameon=False, title_fontsize=10,
          loc="upper center", bbox_to_anchor=(0.5, -0.02),
          ncol=3)

plt.tight_layout()
plt.show()

In [ ]:
print(zones_girec["AREA_TYPE"].value_counts())
print(f"\nNaN : {zones_girec['AREA_TYPE'].isna().sum()}")

In [ ]:
# ─── Walk index moyen par AREA_TYPE ───────────────────────────────────────────
walk_by_area = (
    zones_girec.groupby('AREA_TYPE')['walk_index']
    .agg(['mean', 'median', 'std', 'count'])
    .sort_values('mean', ascending=False)
    .round(4)
)

print(f"{'='*60}")
print(f"Mean walk index by AREA_TYPE — GIREC zones")
print(f"{'='*60}")
print(f"{'AREA_TYPE':<25} {'mean':>8} {'median':>8} {'std':>8} {'n':>6}")
print(f"{'-'*60}")
for area, row in walk_by_area.iterrows():
    print(f"  {area:<23} {row['mean']:>8.4f} {row['median']:>8.4f} {row['std']:>8.4f} {int(row['count']):>6}")
print(f"{'='*60}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))  # ← plus fin
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

# ─── Remapping des labels avec retour à la ligne ──────────────────────────────
mapped_labels = [
    AREA_TYPE_LABELS.get(area, area).replace(' ', '\n') + f"\n(n={int(walk_by_area.loc[area, 'count'])})"
    for area in walk_by_area.index
]
colors = [AREA_TYPE_COLORS.get(area, 'steelblue') for area in walk_by_area.index]

bars = ax.bar(range(len(walk_by_area)), walk_by_area['mean'],
              color=colors, edgecolor='white', linewidth=0.5)

ax.errorbar(range(len(walk_by_area)), walk_by_area['mean'],
            yerr=walk_by_area['std'],
            fmt='none', color='black', capsize=4, linewidth=1)

for bar, val, std, n in zip(bars, walk_by_area['mean'], walk_by_area['std'], walk_by_area['count']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + std + 0.005,
            f'{val:.3f}', #\n(n={int(n)})'
            ha='center', va='bottom', fontsize=12)

ax.set_ylabel('Mean walk index', fontsize=12)
#ax.set_xlabel('Area type', fontsize=12)
ax.set_xticks(range(len(walk_by_area)))
ax.set_xticklabels(mapped_labels, rotation=0, ha='center', fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Densité moyenne par groupe et AREA_TYPE ─────────────────────────────────
area_types = zones_girec["AREA_TYPE"].unique()

# ─── Construire le DataFrame ──────────────────────────────────────────────────
rows = []
for area in area_types:
    zone_subset = zones_girec[zones_girec["AREA_TYPE"] == area]
    row = {"AREA_TYPE": area}
    for key, label in labels_all_groups.items():
        col = f"density_{key}"
        if col in zones_girec.columns:
            vals = zone_subset[col].dropna()
            vals = vals[vals > 0]
            row[label] = vals.mean() if len(vals) > 0 else np.nan
    rows.append(row)

df_area = pd.DataFrame(rows).set_index("AREA_TYPE")

# ─── Trier par walk_index moyen ───────────────────────────────────────────────
area_order = (
    zones_girec.groupby("AREA_TYPE")["walk_index"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
df_area = df_area.loc[area_order]

# ─── Heatmap ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
fig.suptitle("Mean daily pedestrian density by group and area type\n(GIREC zones — sorted by mean walk index)",
             fontsize=13, fontweight="bold")

im = ax.imshow(df_area.values, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(df_area.columns)))
ax.set_yticks(range(len(df_area.index)))
ax.set_xticklabels(df_area.columns, rotation=35, ha='right', fontsize=8)
ax.set_yticklabels(df_area.index, fontsize=9)

# ─── Valeurs dans les cellules ────────────────────────────────────────────────
for i in range(len(df_area.index)):
    for j in range(len(df_area.columns)):
        val = df_area.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.5f}",
                    ha='center', va='center', fontsize=6,
                    color='white' if val > df_area.values[~np.isnan(df_area.values)].max() * 0.6 else 'black')

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01,
             label="Mean daily pedestrian density (legs · day⁻¹ · user⁻¹)")
plt.tight_layout()
plt.show()

# CARTE DE PRIORITE

## CRITICAL AREAS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

col = 'walk_index_norm'

# ── Histogramme ───────────────────────────────────────────────────────────
zones_girec[col].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Count')
axes[0].set_title(f'Distribution — {col} | GIREC')
axes[0].spines[["top", "right"]].set_visible(False)

# Percentiles clés
for p in [75, 90, 95, 99]:
    val = zones_girec[col].quantile(p/100)
    axes[0].axvline(val, linestyle='--', linewidth=0.8, label=f'p{p}={val:.3f}')
axes[0].legend(fontsize=8)

# ── Boxplot ───────────────────────────────────────────────────────────────
zones_girec[col].plot(kind='box', ax=axes[1], color='steelblue')
axes[1].set_ylabel(col)
axes[1].set_title(f'Boxplot — {col} | GIREC')
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

# ── Valeurs numériques + nombre de zones au dessus ────────────────────────
n_total = zones_girec[col].notna().sum()
print(f"\nPercentiles {col} :")
print(f"{'Percentile':<12} {'Value':>8} {'Zones above':>12} {'% above':>8}")
print(f"{'-'*44}")
for p in [50, 75, 90, 95, 99, 100]:
    val        = zones_girec[col].quantile(p/100)
    n_above    = (zones_girec[col] > val).sum()
    pct_above  = 100 * n_above / n_total
    print(f"  p{p:<9} {val:>8.4f} {n_above:>12} {pct_above:>7.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

col = 'density_all_norm'

# ── Histogramme ───────────────────────────────────────────────────────────
zones_girec[col].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Count')
axes[0].set_title(f'Distribution — {col} | GIREC')
axes[0].spines[["top", "right"]].set_visible(False)

# Percentiles clés
for p in [75, 90, 95, 99]:
    val = zones_girec[col].quantile(p/100)
    axes[0].axvline(val, linestyle='--', linewidth=0.8, label=f'p{p}={val:.3f}')
axes[0].legend(fontsize=8)

# ── Boxplot ───────────────────────────────────────────────────────────────
zones_girec[col].plot(kind='box', ax=axes[1], color='steelblue')
axes[1].set_ylabel(col)
axes[1].set_title(f'Boxplot — {col} | GIREC')
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

# ── Valeurs numériques + nombre de zones au dessus ────────────────────────
n_total = zones_girec[col].notna().sum()
print(f"\nPercentiles {col} :")
print(f"{'Percentile':<12} {'Value':>8} {'Zones above':>12} {'% above':>8}")
print(f"{'-'*44}")
for p in [50, 75, 90, 95, 99]:
    val        = zones_girec[col].quantile(p/100)
    n_above    = (zones_girec[col] > val).sum()
    pct_above  = 100 * n_above / n_total
    print(f"  p{p:<9} {val:>8.4f} {n_above:>12} {pct_above:>7.1f}%")

In [ ]:
# Densité vs walk_index
plot_all_quadrants('density_all_norm', 'walk_index', zones_girec,
                   profile_label='All Users',
                   x_label='Density', y_label='Walk index')

In [ ]:
gdf_result = plot_all_quadrants('density_all_norm', 'walk_index', zones_girec,
                                profile_label='All Users',
                                x_label='Density', y_label='Walk index')

plot_quadrant_scatter(gdf_result, x_label='Density (log scale)', y_label='Walk index',
                      x_log_scale=True, show_trend=False)

In [ ]:
print(gdf_result[['_x_norm', '_y_norm', '_quadrant']].describe())

In [ ]:
# Zones critiques uniquement

plot_critical_zones('density_all_norm', 'walk_index', zones_girec,
                   profile_label='All Users',
                   x_label='Density (norm)', y_label='Walk index')

In [ ]:
zones_girec_q = plot_critical_zones(
    'density_all_norm', 'walk_index', zones_girec,
    spatial_level='girec', profile_label='All Users',
    x_label='Density (norm)', y_label='Walk index'
)

# Liste des zones critiques
critical_zones = zones_girec_q[zones_girec_q['_quadrant'] == 'Critical']
print(f"\n{len(critical_zones)} zones critiques")

In [ ]:
# ── Carte 1 : Walk index ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

zones_girec.plot(
    column='walk_index',
    ax=ax,
    cmap='RdYlGn',
    legend=True,
    legend_kwds={'label': 'Walk index', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey'},
    edgecolor='whitesmoke',
    linewidth=0.3
)

# ─── Augmenter la taille de la légende (colorbar) ──────────────────────────
cbar_ax = fig.axes[-1]
cbar_ax.set_ylabel('Walk index', fontsize=12)
cbar_ax.tick_params(labelsize=12)

#critical_zones.plot(ax=ax, color='none', edgecolor='red', linewidth=1.5)
#ax.set_title('Walk index — GIREC\nRed outline = Critical zones', fontsize=11)

ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
p_high = 95 # ← percentile de clip

from matplotlib.colors import LinearSegmentedColormap

cmap_trunc = LinearSegmentedColormap.from_list(
    'magma_trunc',
    plt.cm.magma(np.linspace(0.2, 1.0, 256))  # ← commence à 0.2 au lieu de 0
)

col = 'density_all_norm'
col_clipped = f"{col}_clipped"

v_max = zones_girec[col].quantile(p_high / 100)
zones_girec[col_clipped] = (zones_girec[col].clip(upper=v_max) / v_max).clip(0, 1)

fig, ax = plt.subplots(figsize=(10, 8))

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

zones_girec.plot(
    column=col_clipped,
    ax=ax,
    cmap=cmap_trunc,
    legend=True,
    legend_kwds={'label': f'Density all (P{p_high})', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey'},
    edgecolor='lightgrey',
    linewidth=0.1
)

critical_zones.plot(ax=ax, color='none', edgecolor='red', linewidth=1.5)

# ─── Augmenter la taille de la légende (colorbar) ──────────────────────────
cbar_ax = fig.axes[-1]  # ← le dernier axe créé est celui de la colorbar
cbar_ax.set_ylabel(f'Density all (P{p_high})', fontsize=12)
cbar_ax.tick_params(labelsize=12)
#critical_zones.plot(ax=ax, color='none', edgecolor='red', linewidth=1.5)
#ax.set_title('Pedestrian density (all) — GIREC\nRed outline = Critical zones', fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.show()

zones_girec.drop(columns=[col_clipped], inplace=True)  # nettoyage



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

cmap = cm.get_cmap("magma")
for v in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r, g, b, _ = cmap(v)
    print(f"{v:.2f} → #{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}")

In [ ]:
print(zones_girec.dtypes.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

zones_girec.plot(
    column='density_all_norm',
    ax=ax,
    cmap='plasma',
    vmin=0,
    vmax=1,
    legend=True,
    legend_kwds={'label': 'Density (normalised)', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey'},
    edgecolor='black',
    linewidth=0.3
)

ax.set_title('Pedestrian density (normalised) — GIREC zones', fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p99 = zones_girec['density_all_norm'].quantile(0.99)

# ── Sans clip ─────────────────────────────────────────────────────────────
n, bins, patches = axes[0].hist(zones_girec['density_all_norm'], bins=50, 
                                 color='steelblue', edgecolor='white')
axes[0].axvline(p99, linestyle='--', color='red', linewidth=1, label=f'p99={p99:.3f}')

# Nombre au dessus de chaque barre
for count, x in zip(n, bins):
    if count > 0:
        axes[0].text(x + (bins[1]-bins[0])/2, count + 0.3, str(int(count)),
                     ha='center', va='bottom', fontsize=5)

axes[0].set_xlabel('density_all_norm')
axes[0].set_ylabel('Count')
axes[0].set_title('Sans clip')
axes[0].legend(fontsize=8)
axes[0].spines[["top", "right"]].set_visible(False)

# ── Avec clip à p99 ───────────────────────────────────────────────────────
clipped = zones_girec['density_all_norm'].clip(upper=p99)
n2, bins2, patches2 = axes[1].hist(clipped, bins=50, color='steelblue', edgecolor='white')

for count, x in zip(n2, bins2):
    if count > 0:
        axes[1].text(x + (bins2[1]-bins2[0])/2, count + 0.3, str(int(count)),
                     ha='center', va='bottom', fontsize=5)

axes[1].set_xlabel('density_all_norm (clipped at p99)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Avec clip à p99 ({p99:.3f})')
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle('density_all_norm — effet du clipping à p99 | GIREC', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

p99 = zones_girec['density_all_norm'].quantile(0.99)

# ── Sans clip ─────────────────────────────────────────────────────────────
zones_girec.plot(
    column='density_all_norm',
    ax=axes[0],
    cmap='plasma',
    vmin=0,
    vmax=1,
    legend=True,
    legend_kwds={'label': 'density_all_norm', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey'},
    edgecolor='black',
    linewidth=0.3
)
axes[0].set_title('Sans clip', fontsize=11)
axes[0].set_axis_off()

# ── Avec clip à p99 ───────────────────────────────────────────────────────
zones_girec_clip = zones_girec.copy()
zones_girec_clip['density_clipped'] = zones_girec['density_all_norm'].clip(upper=p99)

zones_girec_clip.plot(
    column='density_clipped',
    ax=axes[1],
    cmap='plasma',
    vmin=0,
    vmax=p99,
    legend=True,
    legend_kwds={'label': f'density_all_norm (clipped p99={p99:.3f})', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey'},
    edgecolor='black',
    linewidth=0.3
)
axes[1].set_title(f'Avec clip à p99 ({p99:.3f})', fontsize=11)
axes[1].set_axis_off()

plt.suptitle('density_all_norm — effet du clipping | GIREC', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Vue rapide par classes
plot_radar_critical_zones(zones_girec, attributs_info, profile='standard',
                           mode='classes', spatial_level='girec',
                           x_low_percentile=25, x_high_percentile=75,
                           y_low_percentile=25, y_high_percentile=75,
                           show_title=False)

# Vue détaillée par attributs groupés par classe
plot_radar_critical_zones(zones_girec, attributs_info, profile='standard',
                           mode='attributes', spatial_level='girec',
                           x_low_percentile=25, x_high_percentile=75,
                           y_low_percentile=25, y_high_percentile=75,
                           show_title=False)

# Zoom sur une zone spécifique
plot_radar_critical_zones(zones_girec, attributs_info, profile='standard',
                           mode='attributes', zone_nom='Les Vernets',
                           x_low_percentile=25, x_high_percentile=75,
                           y_low_percentile=25, y_high_percentile=75,
                           show_title=False)

In [ ]:
# ── Construire la liste des attributs depuis l'Excel ──────────────────────
attrs_df = attributs_info[attributs_info['include_in_index'] == True][['attribute', 'Class']].copy()

print(f"{'='*50}")
print(f"Attributs inclus dans l'index : {len(attrs_df)}")
print(f"{'='*50}")
print(attrs_df.to_string(index=False))

print(f"\n{'='*50}")
print(f"Répartition par classe :")
print(f"{'='*50}")
print(attrs_df['Class'].value_counts().to_string())

In [ ]:
# ── Vérification dans zones_girec ─────────────────────────────────────────
for profile, suffix in [('standard', ''), ('women', '_women'), ('senior', '_senior')]:
    attr_cols = [f"{a}{suffix}" for a in attrs_df['attribute']]
    missing   = [c for c in attr_cols if c not in zones_girec.columns]
    print(f"\nProfil '{profile}' — {len(attr_cols)} attributs attendus")
    if missing:
        print(f"  ⚠ Manquants : {missing}")
    else:
        print(f"  ✓ Tous présents dans zones_girec")

## DESSERT X DENSITY X WALK-INDEX

In [ ]:
print(zones_girec.dtypes.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

zones_girec.plot(
    column='desserte_score',
    ax=ax,
    cmap='RdYlGn',
    legend=True,
    legend_kwds={'label': 'Desserte score', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey'},
    edgecolor='black',
    linewidth=0.3
)
ax.set_title('Transit accessibility score — GIREC', fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

col = 'desserte_score'

# ── Histogramme ───────────────────────────────────────────────────────────
zones_girec[col].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel(col)
axes[0].set_ylabel('Count')
axes[0].set_title(f'Distribution — {col} | GIREC')
axes[0].spines[["top", "right"]].set_visible(False)

for p in [25, 50, 75, 90]:
    val = zones_girec[col].quantile(p/100)
    axes[0].axvline(val, linestyle='--', linewidth=0.8, label=f'p{p}={val:.3f}')
axes[0].legend(fontsize=8)

# ── Boxplot ───────────────────────────────────────────────────────────────
zones_girec[col].plot(kind='box', ax=axes[1], color='steelblue')
axes[1].set_ylabel(col)
axes[1].set_title(f'Boxplot — {col} | GIREC')
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle(f'Distribution — {col} | GIREC', fontsize=12)
plt.tight_layout()
plt.show()

# ── Valeurs numériques + nombre de zones au dessus ────────────────────────
n_total = zones_girec[col].notna().sum()
print(f"\nPercentiles {col} :")
print(f"{'Percentile':<12} {'Value':>8} {'Zones above':>12} {'% above':>8}")
print(f"{'-'*44}")
for p in [25, 50, 75, 90, 95, 99, 100]:
    val       = zones_girec[col].quantile(p/100)
    n_above   = (zones_girec[col] > val).sum()
    pct_above = 100 * n_above / n_total
    print(f"  p{p:<9} {val:>8.4f} {n_above:>12} {pct_above:>7.1f}%")

In [ ]:
desserte_by_area = (
    zones_girec.groupby('AREA_TYPE')['desserte_score']
    .agg(['mean', 'median', 'std', 'count'])
    .sort_values('mean', ascending=False)
    .round(4)
)

print(f"{'='*60}")
print(f"Mean desserte score by AREA_TYPE — GIREC zones")
print(f"{'='*60}")
print(f"{'AREA_TYPE':<25} {'mean':>8} {'median':>8} {'std':>8} {'n':>6}")
print(f"{'-'*60}")
for area, row in desserte_by_area.iterrows():
    print(f"  {area:<23} {row['mean']:>8.4f} {row['median']:>8.4f} {row['std']:>8.4f} {int(row['count']):>6}")
print(f"{'='*60}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))  # ← plus fin pour mise en page côte à côte
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

area_order_desserte = desserte_by_area.index.tolist()
data_by_area = [zones_girec[zones_girec['AREA_TYPE'] == area]['desserte_score'].dropna().values
                for area in area_order_desserte]
colors = [AREA_TYPE_COLORS.get(area, 'steelblue') for area in area_order_desserte]

# ─── Labels avec mapping ──────────────────────────────────────────────────────
tick_labels = [
    AREA_TYPE_LABELS.get(area, area).replace(' ', '\n')
    for area in area_order_desserte
]

bp = ax.boxplot(data_by_area,
                tick_labels=tick_labels,
                patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                flierprops=dict(marker='o', markersize=3, alpha=0.4))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_ylabel('Transit accessibility score', fontsize=11)
ax.tick_params(axis='x', labelsize=9)
ax.tick_params(axis='y', labelsize=10)
ax.spines[["top", "right"]].set_visible(False)
#ax.set_title('Transit accessibility score by urban area type — GIREC zones', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### DESSERTE X DENSITY

In [ ]:
gdf_result_desserte = plot_all_quadrants(
    'density_all', 'desserte_score', zones_girec,
    spatial_level='girec', profile_label='All Users — Transit accessibility',
    x_label='Density', y_label='Transit accessibility',
    x_low_percentile=25, x_high_percentile=75,
    y_low_percentile=25, y_high_percentile=75
)

plot_quadrant_scatter(gdf_result_desserte, x_label='Density (log scale)', y_label='Transit accessibility',
                      x_log_scale=True, show_trend=False)

### DESSERTE X WALKABILITY

In [ ]:
plot_all_quadrants(
    'desserte_score', 'walk_index', zones_girec,
    spatial_level='girec', profile_label='Transit accessibility vs Walk index',
    x_label='Transit accessibility', y_label='Walk index',
    x_normalize=False, y_normalize=False,
    x_low_percentile=25, x_high_percentile=75,
    y_low_percentile=25, y_high_percentile=75
)

In [ ]:
gdf_result_desserte_walk = plot_all_quadrants(
    'desserte_score', 'walk_index', zones_girec,
    spatial_level='girec', profile_label='Transit accessibility vs Walk index',
    x_label='Transit accessibility', y_label='Walk index',
    x_normalize=False, y_normalize=False,
    x_low_percentile=25, x_high_percentile=75,
    y_low_percentile=25, y_high_percentile=75
)

plot_quadrant_scatter(gdf_result_desserte_walk, x_label='Transit accessibility', y_label='Walk index', show_trend=False)

In [ ]:
# Capturer les zones critiques
zones_girec_desserte_q = plot_critical_zones(
    'desserte_score', 'walk_index', zones_girec,
    spatial_level='girec', profile_label='Walk index vs Transit accessibility',
    x_label='Transit accessibility', y_label='Walk index',
    x_normalize=False, y_normalize=False,
    x_low_percentile=25, x_high_percentile=75,
    y_low_percentile=25, y_high_percentile=75
)

# Zones critiques uniquement
critical_desserte = zones_girec_desserte_q[zones_girec_desserte_q['_quadrant'] == 'Critical']

print(f"\nTotal zones critiques : {len(critical_desserte)}")
print(f"\n{'='*50}")
print(f"Répartition par AREA_TYPE :")
print(f"{'='*50}")
area_counts = critical_desserte['AREA_TYPE'].value_counts()
area_pct    = critical_desserte['AREA_TYPE'].value_counts(normalize=True) * 100
for area in area_counts.index:
    print(f"  {area:<25} : {area_counts[area]:>3} zones ({area_pct[area]:.1f}%)")

print(f"\n{'='*50}")
print(f"Walk index moyen par AREA_TYPE (only on zones critiques) :")
print(f"{'='*50}")
print(critical_desserte.groupby('AREA_TYPE')[['walk_index', 'desserte_score']].mean().round(4).to_string())

In [ ]:
# Radar sur les zones critiques
plot_radar_critical_zones(zones_girec, attributs_info, profile='standard',
                           col_x='desserte_score', col_y='walk_index',
                           mode='classes', spatial_level='girec',
                           x_normalize=False, y_normalize=False,
                           x_low_percentile=25, x_high_percentile=75,
                           y_low_percentile=25, y_high_percentile=75, show_title=False)

In [ ]:
plot_radar_critical_zones(zones_girec, attributs_info, profile='standard',
                           col_x='desserte_score', col_y='walk_index',
                           mode='attributes', spatial_level='girec',
                           x_normalize=False, y_normalize=False,
                           x_low_percentile=25, x_high_percentile=75,
                           y_low_percentile=25, y_high_percentile=75, show_title=False)

In [ ]:
plot_radar_critical_zones(zones_girec, attributs_info, profile='standard',
                           col_x='desserte_score', col_y='walk_index',
                           mode='attributes', spatial_level='girec',
                           x_normalize=False, y_normalize=False,
                           x_low_percentile=25, x_high_percentile=75,
                           y_low_percentile=25, y_high_percentile=75,
                           zone_nom='Le-ROYER')

# PRECARIOUSNESS X WALKABILITY

In [ ]:
zones_girec['precarity_area'] = zones_girec['precarite_score_24'].apply(assign_precarity_area)

In [ ]:
zones_girec

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for area, color in PRECARITY_AREA_COLORS.items():
    subset = zones_girec[zones_girec['precarity_area'] == area]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.3)

nan_zones = zones_girec[zones_girec['precarity_area'].isna()]
if len(nan_zones) > 0:
    nan_zones.plot(ax=ax, color='lightgrey', edgecolor='black', linewidth=0.3)

# Légende manuelle
patches = [
    mpatches.Patch(color=color, label=f'{area} (n={len(zones_girec[zones_girec["precarity_area"] == area])})')
    for area, color in PRECARITY_AREA_COLORS.items()
]
if len(nan_zones) > 0:
    patches.append(mpatches.Patch(color='lightgrey', label=f'No data (n={len(nan_zones)})'))

ax.legend(handles=patches, loc='lower right', framealpha=0.9, fontsize=8)
ax.set_title('Precarity area categories — GIREC zones', fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# ─── Répartition des scores de précarité par AREA_TYPE ────────────────────────
df_area = zones_girec[["AREA_TYPE", "precarite_score_24"]].dropna()

cross_score = (df_area
    .groupby(["AREA_TYPE", "precarite_score_24"])
    .size()
    .unstack(fill_value=0)
    .astype(int)
)

# ─── Tableau croisé avec classes regroupées ───────────────────────────────────
df_area2 = df_area.copy()
df_area2['precarity_class'] = df_area2['precarite_score_24'].apply(assign_precarity_area)

cross_class = (df_area2
    .groupby(["AREA_TYPE", "precarity_class"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=PRECARITY_AREA_ORDER, fill_value=0)
)
cross_class_pct = cross_class.div(cross_class.sum(axis=1), axis=0).mul(100).round(1)
cross_score_pct = cross_score.div(cross_score.sum(axis=1), axis=0).mul(100).round(1)

print("── Nombre de sous-secteurs par AREA_TYPE × score précarité ──")
print(cross_score.to_string())

print("\n── Répartition par AREA_TYPE × classe de précarité (n) ──────────")
print(cross_class.to_string())

print("\n── Répartition par AREA_TYPE × classe de précarité (%) ──────────")
print(cross_class_pct.to_string())

print("\n── Total par classe de précarité (toutes zones confondues) ───────")
totals = cross_class.sum(axis=0)
totals_pct = (totals / totals.sum() * 100).round(1)
for cls in PRECARITY_AREA_ORDER:
    print(f"  {cls:<30} n={totals[cls]:>3}  ({totals_pct[cls]:.1f}%)")

# ─── Paramètres ───────────────────────────────────────────────────────────────
score_colors = ["#55A868", "#a8c55a", "#e7c481", "#f39c12", "#e67e22", "#C44E52", "#8B0000"]
score_colors = score_colors[:len(cross_score.columns)]
class_colors = [PRECARITY_AREA_COLORS[c] for c in PRECARITY_AREA_ORDER]
min_height_pct = 4

def add_bar_labels(ax, dataframe, min_height, fmt="{:.0f}"):
    """Ajoute les valeurs dans les portions de barres empilées si assez grandes."""
    cumulative = pd.Series(0.0, index=dataframe.index)
    for col_idx, col in enumerate(dataframe.columns):
        for bar_idx, (area, val) in enumerate(dataframe[col].items()):
            if val >= min_height:
                y_pos = cumulative[area] + val / 2
                ax.text(
                    bar_idx, y_pos,
                    fmt.format(val),
                    ha='center', va='center',
                    fontsize=12, color='black'
                )
        cumulative += dataframe[col]

# ── 1. En % par AREA_TYPE — scores détaillés ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
cross_score_pct.plot(kind="bar", stacked=True, ax=ax,
                     color=score_colors, edgecolor="white", linewidth=0.5)
add_bar_labels(ax, cross_score_pct, min_height_pct, fmt="{:.1f}%")
ax.set_title("% of sub-sectors by zone type", fontweight="bold")
ax.set_xlabel("Zone type (AREA_TYPE)")
ax.set_ylabel("% of sub-sectors")
ax.tick_params(axis='x', rotation=30)
ax.legend(title="Precarity score", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


# ── 2. Classes regroupées en % ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))  # ← resserré
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

cross_class_pct_mapped = cross_class_pct.copy()
cross_class_pct_mapped.index = [
    AREA_TYPE_LABELS.get(i, i).replace(' ', '\n') 
    for i in cross_class_pct.index
]

cross_class_pct_mapped.plot(kind="bar", stacked=True, ax=ax,
                             color=class_colors, edgecolor="black", linewidth=0.5)

add_bar_labels(ax, cross_class_pct_mapped, min_height_pct, fmt="{:.1f}%")

#ax.set_xlabel("Zone type (AREA_TYPE)", fontsize=12)
ax.set_ylabel("% of sub-sectors", fontsize=12)
ax.tick_params(axis='x', rotation=0, labelsize=12)
ax.set_xticklabels(ax.get_xticklabels(), ha='center', fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.legend(title_fontsize=12, fontsize=11,
          loc="upper center", bbox_to_anchor=(0.5, -0.25),
          ncol=2, frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plot_radar_by_area(
    gdf            = zones_girec,
    attributs_info = attributs_info,
    group_col      = 'precarity_area',
    group_values   = ['Precarious (3-4)', 'Highly precarious (5-6)'],
    group_colors   = [PRECARITY_AREA_COLORS['Precarious (3-4)'], 
                      PRECARITY_AREA_COLORS['Highly precarious (5-6)']],
    mode           = 'classes',
    spatial_level  = 'girec'
)

In [ ]:
plot_radar_by_area(
    gdf            = zones_girec,
    attributs_info = attributs_info,
    group_col      = 'precarity_area',
    group_values   = ['Precarious (3-4)', 'Highly precarious (5-6)'],
    group_colors   = [PRECARITY_AREA_COLORS['Precarious (3-4)'], 
                      PRECARITY_AREA_COLORS['Highly precarious (5-6)']],
    mode           = 'attributes',
    spatial_level  = 'girec'
)

# PEOPLE LOCATION

In [ ]:
print(users_GE_walk_regular.dtypes.to_string())

In [ ]:
from shapely import wkt

# ── Convertir en GeoDataFrame en gérant les NaN ───────────────────────────
def safe_wkt_loads(val):
    if pd.isna(val):
        return None
    return wkt.loads(val)

gdf_users = gpd.GeoDataFrame(
    users_GE_walk_regular,
    geometry=users_GE_walk_regular['home_geometry_from_survey'].apply(safe_wkt_loads),
    crs='EPSG:4326'
).to_crs('EPSG:2056')

print(f"Users total                  : {len(gdf_users)}")
print(f"NaN résidence                : {gdf_users.geometry.isna().sum()}")
print(f"Users avec point de résidence: {gdf_users.geometry.notna().sum()}")

# ── Spatial join résidence → AREA_TYPE ────────────────────────────────────
gdf_users_area = gpd.sjoin(
    gdf_users[gdf_users.geometry.notna()],
    zones_girec[['geometry', 'AREA_TYPE', 'walk_index_norm', 'NOM', 'COMMUNE']],
    how='left',
    predicate='within'
)

print(f"\nRépartition résidence par AREA_TYPE :")
print(gdf_users_area['AREA_TYPE'].value_counts())
print(f"\nHors canton GE (NaN AREA_TYPE) : {gdf_users_area['AREA_TYPE'].isna().sum()}")

# ── Résidents dans le canton uniquement ───────────────────────────────────
gdf_users_GE = gdf_users_area[gdf_users_area['AREA_TYPE'].notna()].copy()

print(f"\n{'='*40}")
print(f"Résidents dans le canton GE  : {len(gdf_users_GE)}")
print(f"Frontaliers (hors canton)    : {gdf_users_area['AREA_TYPE'].isna().sum()}")
print(f"\nRépartition par AREA_TYPE :")
print(gdf_users_GE['AREA_TYPE'].value_counts())
print(f"\nEn % :")
print((gdf_users_GE['AREA_TYPE'].value_counts(normalize=True) * 100).round(1))

In [ ]:
#donne le titre de chaque subplot
columns = {
    "gdr":              "Gender",
    "age_fr_grouped":   "Age group",
    "income_class":  "Income",
    "has_car":          "Car owner",
    "tp_level":   "Public Transport subscription level"     
}

In [ ]:
#donne l'ordre des barres pour chaque sublplot
column_orders = {
    "gdr"            : [labels_all_groups[k] for k, _ in gender_filters],
    "age_fr_grouped" : [labels_all_groups[k] for k, _ in age_filters],
    "income_class"   : [labels_all_groups[k] for k, _ in income_filters],
    "has_car"        : [labels_all_groups[k] for k, _ in car_filters],
    "tp_level"       : [labels_all_groups[k] for k, _ in tp_filters],
}

In [ ]:
# Mapping valeurs brutes → labels anglais par colonne, les labels non présents, c'est parce que ils sont déjà en anglais
col_value_maps = {
    "gdr"            : gender_fr_to_en,
    "age_fr_grouped" : age_fr_to_en,
    "tp_level"       : {num: labels_all_groups[key] for key, num in tp_filters} | 
                       {float(num): labels_all_groups[key] for key, num in tp_filters},
}

for col, label in columns.items():
    if col not in gdf_users_GE.columns:
        print(f"⚠ {col} absent")
        continue

    # Traduire les valeurs si un mapping existe
    col_data = gdf_users_GE[col].map(col_value_maps[col]) if col in col_value_maps else gdf_users_GE[col]

    cross = pd.crosstab(
        col_data,
        gdf_users_GE['AREA_TYPE'],
        normalize='index'
    ).round(3) * 100

    available_cols = [c for c in AREA_TYPE_ORDER if c in cross.columns]
    cross = cross[available_cols]

    # Réordonner les lignes
    if col in column_orders:
        order = [r for r in column_orders[col] if r in cross.index]
        cross = cross.reindex(order)

    print(f"\n{'='*60}")
    print(f"Résidence par AREA_TYPE — {label}")
    print(f"{'='*60}")
    print(cross.to_string())
    print()

In [ ]:
# ── Répartition résidence par groupe socio-démographique ──────────────────

n_cols = len(columns)
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 6), sharey=True)
fig.patch.set_alpha(0)
if n_cols == 1:
    axes = [axes]

for ax, (col, label) in zip(axes, columns.items()):
    ax.patch.set_alpha(0)

    if col not in gdf_users_GE.columns:
        print(f"⚠ {col} absent")
        ax.set_visible(False)
        continue

    # Traduire les valeurs si un mapping existe
    col_data = gdf_users_GE[col].map(col_value_maps[col]) if col in col_value_maps else gdf_users_GE[col]

    cross = pd.crosstab(
        col_data,
        gdf_users_GE['AREA_TYPE'],
        normalize='index'
    ).round(3) * 100

    available_cols = [c for c in AREA_TYPE_ORDER if c in cross.columns]
    cross = cross[available_cols]

    # Réordonner les lignes
    if col in column_orders:
        order = [r for r in column_orders[col] if r in cross.index]
        cross = cross.reindex(order)

    cross.plot(
        kind='bar',
        ax=ax,
        color=[AREA_TYPE_COLORS[c] for c in available_cols],
        edgecolor='white',
        linewidth=0.5,
        legend=False
    )

    ax.set_title(label, fontsize=14, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('% of residents' if ax == axes[0] else '', fontsize=14)
    ax.tick_params(axis='x', rotation=30, labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 100)

# ─── Mapping des noms de typologie territoriale ────────────────────────────
patches = [
    mpatches.Patch(color=AREA_TYPE_COLORS[a],
                   label=AREA_TYPE_LABELS.get(a, a.title()))
    for a in AREA_TYPE_ORDER
]
fig.legend(handles=patches, loc='lower center', ncol=len(AREA_TYPE_ORDER),
           fontsize=14, bbox_to_anchor=(0.5, -0.08), framealpha=0)

# fig.suptitle('Residential location by socio-demographic group',
#              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# TEMPORAL ANALYSIS

In [ ]:
# ─── Courbe temporelle — densité moyenne par créneau horaire ─────────────────
slot_names_only = [s for s, _, _, _, _ in time_slots if s != "all_day"]

mean_density = [
    np.nanmean(rasters_hourly[s][canton_GE_mask]) 
    for s in slot_names_only
]

# ─── Labels courts pour l'axe x ───────────────────────────────────────────────
x_labels = [slot_labels.get(s, s).split("\n")[0] for s in slot_names_only]
x        = np.arange(len(slot_names_only))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x, mean_density, marker='o', color='#4C72B0', linewidth=2)

# ─── Valeurs au-dessus des points ─────────────────────────────────────────────
# for xi, val in zip(x, mean_density):
#     ax.text(xi, val * 1.02, f"{val:.6f}", ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=9)
ax.set_xlabel("Time slot")
ax.set_ylabel("Pixel Density")
ax.set_title(
    f"Evolution of Mean Daily Pedestrian Density by time slot\n"
    f"All users " #| {norm_mode} | P{clip_percentile}"
)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ─── Print des valeurs ────────────────────────────────────────────────────────
print(f"\n── Mean density by time slot ──────────────────────────")
for s, val in zip(slot_names_only, mean_density):
    label = slot_labels.get(s, s).split("\n")[0]
    print(f"  {label:<20} : {val:.8f}")

In [ ]:
# ─── Courbe temporelle heure par heure ────────────────────────────────────────
mean_density_h = [
    np.nanmean(rasters_hourly_h[s][canton_GE_mask])
    for s in hourly_names
]
hours = [int(s[:2]) for s in hourly_names]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(hours, mean_density_h, marker='o', color='#4C72B0', linewidth=2)
# for h, val in zip(hours, mean_density_h):
#     ax.text(h, val * 1.02, f"{val:.6f}", ha='center', va='bottom', fontsize=7)
ax.set_xticks(hours)
ax.set_xticklabels([f"{h:02d}h" for h in hours], fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.set_xlabel("Hour", fontsize=12)
ax.set_ylabel("Mean pixel density \n (legs · day⁻¹ · user⁻¹)", fontsize=12)
#ax.set_title("Evolution of Mean Daily Pedestrian Density — Hourly\nAll users")
ax.grid(alpha=0.3)

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

plt.tight_layout()
#plt.savefig("density_hourly_curve.png", dpi=300, transparent=True)
plt.show()

In [ ]:
# ─── GIF ──────────────────────────────────────────────────────────────────────
import matplotlib.animation as animation
import base64
from IPython.display import HTML

gif_path = output_file_path_PL + f"pedestrian_density_hourly_{norm_mode}_P{clip_percentile}.gif"
fps      = 2

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.patch.set_facecolor("white")

for ax in axes:
    ax.set_facecolor("whitesmoke")
    ax.set_axis_off()

axes[0].set_title("Canton de Genève", color="black", fontsize=12, pad=8)
axes[1].set_title(f"Zoom — {focus_commune_name}", color="black", fontsize=12, pad=8)

first             = hourly_names[0]
density_norm_init = normalize_raster(rasters_hourly_h[first], hourly_h_max, mode=norm_mode)

im_canton = axes[0].imshow(
    density_norm_init, origin='upper', extent=extent,
    cmap=cmap_plot, vmin=0.001, vmax=1, animated=True
)
im_zoom = axes[1].imshow(
    density_norm_init, origin='upper', extent=extent,
    cmap=cmap_plot, vmin=0.001, vmax=1, animated=True
)

axes[1].set_xlim(xmin_z - margin, xmax_z + margin)
axes[1].set_ylim(ymin_z - margin, ymax_z + margin)

for ax in axes:
    canton_GE.boundary.plot(ax=ax, color='black', linewidth=0.8)
    zones_girec.boundary.plot(ax=ax, color='gray', linewidth=0.3, alpha=0.3)
focus_commune.boundary.plot(ax=axes[1], color='red', linewidth=1.5)

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap=cmap_plot, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax,
                    label=f"Mean Daily Pedestrian Density (P{clip_percentile})")

title = fig.suptitle("", color="black", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 0.91, 0.93])

def update(slot_name):
    density_norm = normalize_raster(rasters_hourly_h[slot_name], hourly_h_max, mode=norm_mode)
    im_canton.set_data(density_norm)
    im_zoom.set_data(density_norm)
    title.set_text(
        f"Mean Daily Pedestrian Density — {hourly_labels[slot_name]}\n"
        f"All users | {norm_mode} | P{clip_percentile}"
    )
    return im_canton, im_zoom, title

print("Generating GIF...")
ani = animation.FuncAnimation(
    fig, update,
    frames=hourly_names,
    interval=1000 // fps,
    blit=False
)

ani.save(gif_path, writer="pillow", fps=fps, dpi=100)
plt.close(fig)
print(f"✓ GIF saved : {gif_path}")

with open(gif_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")

HTML(f'<img src="data:image/gif;base64,{b64}" style="max-width:900px"/>')


# EXPORTS

In [ ]:
zones_girec.to_crs(target_crs).to_file(os.path.join(output_file_path_PL, "girec_aggregated_index_density.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_file_path_PL}girec_aggregated_index_density.parquet')
zones_girec.to_csv(f'{output_step3_path}step3_aggregated_index_girec_density.csv')
print("zones_girec_density exported as .gpkg, .parquet, .csv")

carreau_200.to_crs(target_crs).to_file(os.path.join(output_file_path_PL, "carreau_200_aggregated_index_density.gpkg"), driver="GPKG")
carreau_200.to_crs(target_crs).to_parquet(f'{output_file_path_PL}carreau_200_aggregated_index_density.parquet')
print("carreau_200_density exported as .gpkg, .parquet")